# M8: Cosmos Reason LoRA SFT — Supervised Fine-Tuning

**Stage 7: Model Training — parameter-efficient SFT of a driving VLM**

| Item | Detail |
|------|--------|
| **Input** | `s3://av30lab-user-workspace-{account_id}-{region}/users/{profile}/m1/manifest.json` + `datasets/nuscenes-mini/` (shared, read-only) |
| **Output** | `s3://av30lab-user-workspace-{account_id}-{region}/users/{profile}/m8/` |
| **Model** | [nvidia/Cosmos-Reason1-7B](https://huggingface.co/nvidia/Cosmos-Reason1-7B) (8.33 B params) |
| **Method** | LoRA / PEFT (r=16) — frozen vision tower, frozen base weights |
| **Instance** | `ml.g6.24xlarge` (4× L4) — **measured** to train at native 1600×900 |
| **Recipe reference** | [cosmos-cookbook · reason1 / av_video_caption_vqa](https://github.com/nvidia-cosmos/cosmos-cookbook/tree/main/docs/recipes/post_training/reason1/av_video_caption_vqa) |

---

## What this module is — and what it is not

The blog's Stage 7 is **model training**. Its worked example trains **Alpamayo**
(the VLA driving policy). This module trains **Cosmos Reason** (the VLM) instead,
and it is worth being precise about why:

- **Alpamayo has no LoRA path.** Neither `NVlabs/alpamayo1.5` nor the cookbook
  ships an adapter recipe for it, and a full fine-tune of the 10 B policy needs
  **123.56 GiB** of optimizer + gradient + weight state — more than the 88 GiB a
  `g6.24xlarge` has in total. ZeRO-2 does not help: it shards optimizer state and
  gradients, not weights.
- **The cookbook's own Reason1 SFT is a full fine-tune**, which wants 4× 80 GB.
  What fits on this workshop's hardware is **LoRA**, so that is what runs here.

So: this is **parameter-efficient SFT of a real NVIDIA driving VLM on real
nuScenes labels** — not a reproduction of NVIDIA's full-SFT recipe, and not
Alpamayo training. M9 still runs Alpamayo **inference** with NVIDIA's released
checkpoint; the adapter trained here is not fed into it.

## Why the training labels are not circular

A fine-tune is only meaningful if its targets come from outside the model being
trained. This module therefore **does not use M2's captions** — those are
Cosmos Reason's own output, so training on them would teach the model to
reproduce itself and any loss curve would be self-referential.

Instead the targets are **human-authored nuScenes annotation**:

| Target | Source | Who produced it |
|---|---|---|
| Scene description | `scene.json` → `description` | nuScenes human annotators |
| Road-user categories | `sample_annotation` → `instance` → `category.name` | nuScenes human annotators |
| Object counts | the same annotation records | nuScenes human annotators |

**Honest scope note.** nuScenes 3D boxes are annotated over the *full sensor
suite*, not per-camera, so a category present in the labels is not guaranteed to
be visible in the forward camera crop we feed the model. The questions below are
therefore phrased at the **scenario** level ("what appears in this driving
scenario"), which is what the labels actually support, and annotations are
filtered to the higher `visibility` bins to reduce mismatch.


In [ ]:
"""Environment Setup"""
import json
import os
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import boto3

# --- S3 Path Configuration ---
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
AWS_REGION = boto3.session.Session().region_name  # buckets are per-region (see
# infra/av30_constructs/storage.py): the fallback name MUST carry the region, or a
# notebook whose env vars are missing silently derives a bucket that does not exist.
PROFILE = os.environ.get("USER_PROFILE", "default")

USER_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}-{AWS_REGION}")
SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}-{AWS_REGION}")
INPUT_PREFIX = f"users/{PROFILE}/m1/"          # M1's manifest (which frames to use)
OUTPUT_PREFIX = f"users/{PROFILE}/m8/"
NUSCENES_PREFIX = "datasets/nuscenes-mini/"    # shared, read-only — the LABELS
MODEL_CACHE_PREFIX = "model-cache/cosmos-reason1/"

LOCAL_MODEL_DIR = Path("/tmp/cosmos-reason1-7b")
LOCAL_DATA_DIR = Path("/tmp/nuscenes-mini")
LOCAL_OUT_DIR = Path("/tmp/m8-out")
for _d in (LOCAL_MODEL_DIR, LOCAL_DATA_DIR, LOCAL_OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3")

print(f"Account ID:    {ACCOUNT_ID}")
print(f"Profile:       {PROFILE}")
print(f"M1 manifest:   s3://{USER_BUCKET}/{INPUT_PREFIX}manifest.json")
print(f"nuScenes:      s3://{SHARED_BUCKET}/{NUSCENES_PREFIX}")
print(f"Model cache:   s3://{SHARED_BUCKET}/{MODEL_CACHE_PREFIX}")
print(f"Output:        s3://{USER_BUCKET}/{OUTPUT_PREFIX}")


In [ ]:
"""Pre-flight — GPU layout decides the training resolution (measured, not guessed)

MEASURED on ml.g6.24xlarge (4x NVIDIA L4, 22.04 GiB each, 88.16 GiB aggregate),
torch 2.12.1+cu130 / transformers 4.51.3, LoRA r=16 + frozen ViT + gradient
checkpointing (use_reentrant=False), weights sharded by device_map="auto":

    resolution              seq_len   worst-GPU peak   headroom
    native 1600x900 (1.44MP)   1880      11.28 GiB     10.76 GiB   <- fits
    1.00 MP                    1352       9.47 GiB     12.57 GiB
    0.60 MP                     776       7.35 GiB     14.69 GiB

Weights alone are 15.61 GiB, split [3.16, 4.40, 4.40, 3.65] across the 4 cards.
That is also why ONE 24 GB card fails: 15.6 of its 22.04 GiB is weights, leaving
~6.4 GiB, and the forward pass at native resolution needs far more than that.
(Measured: OOM at 21.54 GiB on a single L4.)
"""
import subprocess

def probe_gpus():
    """Per-GPU (total_mib, used_mib) via nvidia-smi — creates NO CUDA context.

    Touching torch.cuda here would pin a context inside the Jupyter kernel for the
    whole session, stealing memory from the training run itself.
    """
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.total,memory.used",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True).stdout
    return [tuple(int(x.strip()) for x in line.split(","))
            for line in out.strip().splitlines() if line.strip()]

try:
    _gpus = probe_gpus()
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError(
        "nvidia-smi unavailable — this looks like the CPU image. Open Instance "
        "Options and pick a GPU instance (ml.g6.24xlarge), then re-run."
    ) from e

GPU_COUNT = len(_gpus)
per_gpu_gb = [t / 1024 for t, _ in _gpus]
MIN_PER_GPU_GB = min(per_gpu_gb)
MAX_PER_GPU_GB = max(per_gpu_gb)
print(f"GPUs: {GPU_COUNT}")
for i, (t, u) in enumerate(_gpus):
    print(f"  GPU {i}: {t/1024:.1f} GB total — {u} MiB in use")

# Halt on foreign occupancy: another kernel holding VRAM turns into an opaque OOM
# deep inside the backward pass, 15+ minutes into the run.
_busy = [(i, u) for i, (_, u) in enumerate(_gpus) if u > 1024]
if _busy:
    raise SystemExit(
        "GPU memory already in use by another process: "
        + ", ".join(f"GPU {i}: {u} MiB" for i, u in _busy)
        + "\n\nFix: JupyterLab menu -> Kernel -> Shut Down All Kernels, confirm with"
          "\n  nvidia-smi --query-gpu=memory.used --format=csv,noheader"
          "\nthen Run All Cells again."
    )

# --- pick the resolution tier -------------------------------------------------
if MAX_PER_GPU_GB >= 40:
    MAX_PIXELS, TIER = 1600 * 900, "single GPU >=40 GB — native resolution"
elif GPU_COUNT >= 4 and MIN_PER_GPU_GB >= 21:
    MAX_PIXELS, TIER = 1600 * 900, "4x 24 GB — native resolution (MEASURED path)"
elif GPU_COUNT >= 2:
    MAX_PIXELS, TIER = 768 * 768, "2-3 GPUs — 0.60 MP (conservative, NOT measured here)"
else:
    raise RuntimeError(
        f"Only {GPU_COUNT} GPU with {MIN_PER_GPU_GB:.0f} GB. Cosmos-Reason1-7B's "
        f"weights alone are 15.61 GiB, so a single 24 GB card leaves ~6.4 GiB and "
        f"OOMs in the forward pass (measured: 21.54 GiB requested). Pick a "
        f"multi-GPU box (ml.g6.24xlarge) or one card >=40 GB (ml.g7e.2xlarge) in "
        f"Instance Options."
    )
print(f"\nResolution tier: {TIER}")
print(f"max_pixels = {MAX_PIXELS} ({MAX_PIXELS/1e6:.2f} MP)")


In [ ]:
"""Install PEFT (LoRA). --no-deps so pip cannot swap the image's CUDA-matched torch."""
try:
    import peft  # noqa: F401
    print(f"peft already present: {peft.__version__}")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "peft==0.14.0"], check=True)
    import peft
    print(f"peft installed: {peft.__version__}")

import torch
import transformers
print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"CUDA devices {torch.cuda.device_count()}")


In [ ]:
"""Build the VQA training set from HUMAN nuScenes annotation (never from M2)."""

# 1. Which frames? M1 decided. This is the real M1 -> M8 dependency.
try:
    _m = s3.get_object(Bucket=USER_BUCKET, Key=f"{INPUT_PREFIX}manifest.json")
    m1_manifest = json.loads(_m["Body"].read())
except s3.exceptions.NoSuchKey:
    raise SystemExit(
        f"M1 output not found at s3://{USER_BUCKET}/{INPUT_PREFIX}manifest.json.\n"
        f"Run M1 (Data Exploration) first — it selects the frames this module trains on."
    )
cam_files = m1_manifest["cam_front_files"]
cam_scenes = m1_manifest["cam_front_scenes"]
print(f"M1 selected {len(cam_files)} CAM_FRONT key frames across "
      f"{len(set(cam_scenes))} scenes")

# 2. Pull nuScenes metadata + images (read-only shared bucket).
print("\nSyncing nuScenes-mini from S3 ...")
subprocess.run(["aws", "s3", "sync", f"s3://{SHARED_BUCKET}/{NUSCENES_PREFIX}",
                str(LOCAL_DATA_DIR), "--quiet"], check=True)

meta_dir = next((p.parent for p in LOCAL_DATA_DIR.rglob("scene.json")), None)
if meta_dir is None:
    raise SystemExit("scene.json not found under the synced dataset — is it staged?")
tables = {p.stem: json.loads(p.read_text()) for p in meta_dir.glob("*.json")}
print(f"Loaded {len(tables)} metadata tables from {meta_dir}")

# 3. The joins. nuScenes stores NO category on the annotation and NO channel on
#    sample_data, so both must be resolved through link tables:
#      sample_annotation -> instance -> category.name
#      sample_data -> calibrated_sensor -> sensor.channel
for _t in ("category", "instance", "scene", "sample_data", "sample_annotation"):
    if _t not in tables:
        raise SystemExit(f"nuScenes table {_t}.json is missing from the staged "
                         f"dataset — re-run scripts/stage_nuscenes.sh (admin).")

category_by_token = {c["token"]: c["name"] for c in tables["category"]}
instance_by_token = {i["token"]: i for i in tables["instance"]}
scene_by_name = {s["name"]: s for s in tables["scene"]}
sd_by_filename = {sd["filename"]: sd for sd in tables["sample_data"]}

# visibility_token "1".."4" = 0-40 / 40-60 / 60-80 / 80-100 % visible.
# Keep the two highest bins so a labelled object is likely actually in view.
VISIBLE_BINS = {"3", "4"}

ann_by_sample = {}
for a in tables["sample_annotation"]:
    ann_by_sample.setdefault(a["sample_token"], []).append(a)

def short(cat: str) -> str:
    """'vehicle.car' -> 'car'; 'human.pedestrian.adult' -> 'pedestrian'."""
    parts = cat.split(".")
    return parts[1] if len(parts) > 2 else parts[-1]

def categories_for(sample_token: str):
    out = Counter()
    for a in ann_by_sample.get(sample_token, []):
        if a.get("visibility_token") not in VISIBLE_BINS:
            continue
        inst = instance_by_token.get(a.get("instance_token"), {})
        name = category_by_token.get(inst.get("category_token"))
        if name:
            out[short(name)] += 1
    return out

# 4. Emit QA pairs. Three question types, all answered from human labels.
examples = []
for fname, scene_name in zip(cam_files, cam_scenes):
    sd = sd_by_filename.get(fname)
    scene = scene_by_name.get(scene_name)
    if sd is None or scene is None:
        continue
    img_path = meta_dir.parent / fname
    if not img_path.exists():
        continue
    cats = categories_for(sd["sample_token"])
    if not cats:
        continue
    listing = ", ".join(f"{n} {c}" for c, n in cats.most_common())
    examples.append({
        "image": str(img_path), "scene": scene_name,
        "qa": [
            ("Describe this driving scenario.", scene["description"]),
            ("What road users appear in this scenario, and how many of each?", listing),
            ("How many annotated road users are around the vehicle?",
             str(sum(cats.values()))),
        ],
    })

# Flatten to one (image, question, answer) row per QA pair.
rows = [{"image": e["image"], "scene": e["scene"], "question": q, "answer": a}
        for e in examples for q, a in e["qa"]]

# Hold out whole SCENES, not rows — holding out rows would leak, because the same
# frame appears under three questions.
scenes_sorted = sorted({e["scene"] for e in examples})
HELD_OUT = set(scenes_sorted[-2:]) if len(scenes_sorted) > 3 else set(scenes_sorted[-1:])
train_rows = [r for r in rows if r["scene"] not in HELD_OUT]
eval_rows = [r for r in rows if r["scene"] in HELD_OUT]

print(f"\nFrames usable:  {len(examples)}")
print(f"QA rows total:  {len(rows)}")
print(f"  train:        {len(train_rows)}  ({len(scenes_sorted) - len(HELD_OUT)} scenes)")
print(f"  held out:     {len(eval_rows)}  (scenes: {', '.join(sorted(HELD_OUT))})")
print("\nExample row:")
print(json.dumps({k: v for k, v in train_rows[0].items()}, indent=2)[:600])

(LOCAL_OUT_DIR / "dataset.json").write_text(json.dumps(
    {"train": train_rows, "eval": eval_rows,
     "label_source": "nuScenes human annotation (scene.description, "
                     "sample_annotation->instance->category)",
     "visibility_bins_kept": sorted(VISIBLE_BINS)}, indent=2))


In [ ]:
"""Load Cosmos-Reason1-7B from the S3 cache, sharded across the GPUs."""
from transformers import AutoModelForImageTextToText, AutoProcessor

print(f"Restoring model from s3://{SHARED_BUCKET}/{MODEL_CACHE_PREFIX} ...")
t0 = time.time()
subprocess.run(["aws", "s3", "sync", f"s3://{SHARED_BUCKET}/{MODEL_CACHE_PREFIX}",
                str(LOCAL_MODEL_DIR), "--exclude", ".cache/*", "--quiet"], check=True)
download_time = time.time() - t0
print(f"Model restored in {download_time:.0f}s")

# transformers renamed this kwarg (torch_dtype -> dtype). Accept either so the
# notebook does not break when the Distribution image's version moves.
_load_kw = dict(attn_implementation="sdpa", device_map="auto", trust_remote_code=True)
t0 = time.time()
try:
    model = AutoModelForImageTextToText.from_pretrained(
        str(LOCAL_MODEL_DIR), dtype=torch.bfloat16, **_load_kw)
except TypeError:
    model = AutoModelForImageTextToText.from_pretrained(
        str(LOCAL_MODEL_DIR), torch_dtype=torch.bfloat16, **_load_kw)
load_time = time.time() - t0
model.config.use_cache = False          # incompatible with gradient checkpointing

processor = AutoProcessor.from_pretrained(
    str(LOCAL_MODEL_DIR), trust_remote_code=True,
    min_pixels=256 * 28 * 28, max_pixels=MAX_PIXELS)

print(f"Loaded in {load_time:.0f}s")
print(f"Base params: {sum(p.numel() for p in model.parameters()):,}")
weights_per_gpu = [torch.cuda.memory_allocated(i) / 1024**3
                   for i in range(torch.cuda.device_count())]
print(f"Weights per GPU (GiB): {[round(x, 2) for x in weights_per_gpu]} "
      f"= {sum(weights_per_gpu):.2f} total")
print(f"device_map entries: {len(getattr(model, 'hf_device_map', {}) or {})}")


In [ ]:
"""Attach LoRA, freeze the vision tower, turn on gradient checkpointing."""
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    # Adapt the LANGUAGE model only. Adapting the ViT as well roughly doubles
    # activation memory for no measured benefit on this task.
    exclude_modules=r".*visual.*",
)
model = get_peft_model(model, lora_cfg)
for name, p in model.named_parameters():
    if "visual" in name:
        p.requires_grad_(False)

# use_reentrant=False is REQUIRED here: the default reentrant checkpoint silently
# no-ops when no input tensor requires grad, which is exactly the case with a
# frozen base model + LoRA. Getting this wrong looks like "checkpointing enabled"
# while memory behaves as if it were off.
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

gc_on = [n or "<root>" for n, m in model.named_modules()
         if getattr(m, "gradient_checkpointing", False)]
assert gc_on, "gradient checkpointing did not engage — training will OOM"

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Checkpointing active in: {gc_on}")
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")


In [ ]:
"""Train. Small step budget on purpose — this demonstrates the mechanics and
cost of PEFT SFT, it does not chase a converged model."""
import random

from PIL import Image

NUM_STEPS = int(os.environ.get("M8_STEPS", "40"))
GRAD_ACCUM = 4
LR = 1e-4
SEED = 0

random.seed(SEED)
torch.manual_seed(SEED)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR)
first_device = next(model.parameters()).device

def build_batch(row, *, for_generation=False):
    """Encode one row.

    for_generation=False -> teacher-forced: the answer is in the sequence and the
        labels are the sequence with padding masked out.
    for_generation=True  -> the answer is withheld and the chat template appends
        the assistant generation prompt, so model.generate() continues from there.
        (Feeding an EMPTY assistant turn instead would make the model condition on
        a closed, empty answer — a different distribution than inference.)
    """
    img = Image.open(row["image"]).convert("RGB")
    msgs = [{"role": "user", "content": [{"type": "image"},
                                         {"type": "text", "text": row["question"]}]}]
    if not for_generation:
        msgs.append({"role": "assistant",
                     "content": [{"type": "text", "text": row["answer"]}]})
    text = processor.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=for_generation)
    batch = processor(text=[text], images=[img], return_tensors="pt", padding=True)
    batch = {k: v.to(first_device) for k, v in batch.items()}
    if not for_generation:
        labels = batch["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        batch["labels"] = labels
    return batch

model.train()
history, order = [], list(range(len(train_rows)))
random.shuffle(order)
t_start = time.time()

for step in range(NUM_STEPS):
    optimizer.zero_grad(set_to_none=True)
    for _i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(_i)   # per-STEP peak, not run-global
    step_loss = 0.0
    for k in range(GRAD_ACCUM):
        row = train_rows[order[(step * GRAD_ACCUM + k) % len(order)]]
        out = model(**build_batch(row))
        (out.loss / GRAD_ACCUM).backward()
        step_loss += out.loss.item() / GRAD_ACCUM
    optimizer.step()
    peak = max(torch.cuda.max_memory_allocated(i) / 1024**3
               for i in range(torch.cuda.device_count()))
    history.append({"step": step + 1, "loss": round(step_loss, 4),
                    "worst_gpu_peak_gib": round(peak, 2),
                    "elapsed_s": round(time.time() - t_start, 1)})
    if step % 5 == 0 or step == NUM_STEPS - 1:
        print(f"step {step+1:3}/{NUM_STEPS}  loss {step_loss:7.4f}  "
              f"worst-GPU peak {peak:5.2f} GiB  {time.time()-t_start:6.1f}s")

train_time = time.time() - t_start
first5 = sum(h["loss"] for h in history[:5]) / 5
last5 = sum(h["loss"] for h in history[-5:]) / 5
print(f"\nTrained {NUM_STEPS} steps x {GRAD_ACCUM} accum in {train_time/60:.1f} min")
print(f"Mean loss: first 5 steps {first5:.4f} -> last 5 steps {last5:.4f} "
      f"({100*(first5-last5)/first5:+.1f}%)")


In [ ]:
"""Loss curve — plotted from the measured history above, not a stock image."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

steps = [h["step"] for h in history]
losses = [h["loss"] for h in history]
peaks = [h["worst_gpu_peak_gib"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(steps, losses, marker="o", ms=3, color="steelblue")
ax1.set_xlabel("optimizer step"); ax1.set_ylabel("loss")
ax1.set_title(f"M8 LoRA SFT loss ({len(train_rows)} train rows)")
ax1.grid(alpha=.3)

ax2.plot(steps, peaks, color="darkorange")
ax2.axhline(MIN_PER_GPU_GB, ls="--", color="crimson",
            label=f"card capacity {MIN_PER_GPU_GB:.1f} GiB")
ax2.set_xlabel("optimizer step"); ax2.set_ylabel("worst-GPU peak (GiB)")
ax2.set_title("Peak memory vs card capacity"); ax2.legend(); ax2.grid(alpha=.3)

plt.tight_layout()
plt.savefig(LOCAL_OUT_DIR / "loss_curve.png", dpi=100, bbox_inches="tight")
display(IPImage(filename=str(LOCAL_OUT_DIR / "loss_curve.png")))


In [ ]:
"""Held-out check: adapter ON vs OFF on scenes the model never trained on.

PEFT lets us toggle the adapter on the SAME loaded weights, so this is a genuine
A/B of the fine-tune rather than two different models.
"""
import contextlib

model.eval()
samples = eval_rows[:3]
eval_out = []

for row in samples:
    batch = build_batch(row, for_generation=True)
    gen = {}
    for label, disabled in (("base (adapter off)", True), ("sft (adapter on)", False)):
        # model.disable_adapter() is a context manager (PEFT's documented way to run
        # the frozen base). Driving it with `with` rather than __enter__/__exit__ by
        # hand matters: a manual __exit__(None, None, None) tells the context manager
        # the block succeeded even when generate() raised, so a real failure could be
        # swallowed and the adapter left in the wrong state for the next iteration.
        with (model.disable_adapter() if disabled else contextlib.nullcontext()):
            with torch.no_grad():
                ids = model.generate(**batch, max_new_tokens=64, do_sample=False)
            gen[label] = processor.tokenizer.decode(
                ids[0][batch["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    eval_out.append({"scene": row["scene"], "question": row["question"],
                     "human_label": row["answer"], **gen})

for e in eval_out:
    print("=" * 72)
    print(f"scene {e['scene']}  |  Q: {e['question']}")
    print(f"  human label : {e['human_label'][:150]}")
    print(f"  base        : {e['base (adapter off)'][:150]}")
    print(f"  sft         : {e['sft (adapter on)'][:150]}")
print("=" * 72)
print("\nA 40-step LoRA run on ~dozens of rows will NOT transform the outputs. What\n"
      "this shows is the mechanism working end to end: real labels in, adapter\n"
      "trained, A/B reproducible. Scaling steps/data is the knob, not the code.")

(LOCAL_OUT_DIR / "eval_samples.json").write_text(json.dumps(eval_out, indent=2))


In [ ]:
"""Save the adapter + the measured training record, and upload to S3."""
adapter_dir = LOCAL_OUT_DIR / "lora_adapter"
model.save_pretrained(str(adapter_dir))       # adapter only — a few dozen MB
adapter_bytes = sum(f.stat().st_size for f in adapter_dir.rglob("*") if f.is_file())

training_log = {
    "module": "M8_Cosmos_Reason_SFT",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": "nvidia/Cosmos-Reason1-7B",
    "method": {"type": "LoRA", "r": lora_cfg.r, "alpha": lora_cfg.lora_alpha,
               "dropout": lora_cfg.lora_dropout,
               "target_modules": list(lora_cfg.target_modules),
               "excluded": lora_cfg.exclude_modules,
               "gradient_checkpointing": "use_reentrant=False",
               "vision_tower": "frozen"},
    "hardware": {"gpu_count": GPU_COUNT,
                 "per_gpu_gb": [round(x, 2) for x in per_gpu_gb],
                 "resolution_tier": TIER, "max_pixels": MAX_PIXELS},
    "params": {"total": int(total), "trainable": int(trainable),
               "trainable_pct": round(100 * trainable / total, 4)},
    "data": {"label_source": "nuScenes human annotation (NOT M2 captions)",
             "train_rows": len(train_rows), "eval_rows": len(eval_rows),
             "held_out_scenes": sorted(HELD_OUT)},
    "run": {"steps": NUM_STEPS, "grad_accum": GRAD_ACCUM, "lr": LR, "seed": SEED,
            "train_minutes": round(train_time / 60, 2),
            "loss_first5_mean": round(first5, 4), "loss_last5_mean": round(last5, 4)},
    "history": history,
    "adapter_bytes": adapter_bytes,
}
(LOCAL_OUT_DIR / "training_log.json").write_text(json.dumps(training_log, indent=2))

uploaded = []
for f in sorted(LOCAL_OUT_DIR.rglob("*")):
    if not f.is_file():
        continue
    key = f"{OUTPUT_PREFIX}{f.relative_to(LOCAL_OUT_DIR).as_posix()}"
    s3.upload_file(str(f), USER_BUCKET, key)
    uploaded.append((key, f.stat().st_size))

print(f"Adapter size: {adapter_bytes/1024**2:.1f} MiB "
      f"(vs {sum(weights_per_gpu):.1f} GiB of base weights)")
print(f"\nUploaded {len(uploaded)} objects to s3://{USER_BUCKET}/{OUTPUT_PREFIX}")
for key, size in uploaded:
    print(f"  {size/1024:9.1f} KiB  {key}")


In [ ]:
"""Cost Analysis — from the instance metadata and the measured run time."""
INSTANCE_RATES = {
    "ml.g5.12xlarge": 7.09, "ml.g5.24xlarge": 10.18, "ml.g5.48xlarge": 20.36,
    "ml.g6.12xlarge": 5.752, "ml.g6.24xlarge": 8.344, "ml.g6.48xlarge": 16.688,
    # g7e = RTX PRO 6000 Blackwell, 96 GB PER CARD (1 GPU on 2xl/4xl/8xl,
    # 2 on 12xl, 4 on 24xl, 8 on 48xl). 96 GB clears the top tier on ONE card.
    "ml.g7e.2xlarge": 4.2039, "ml.g7e.4xlarge": 4.9977, "ml.g7e.8xlarge": 6.5853,
    "ml.g7e.12xlarge": 10.3576, "ml.g7e.24xlarge": 20.7152, "ml.g7e.48xlarge": 41.4304,
    "ml.p4d.24xlarge": 25.251286, "ml.p5.48xlarge": 63.296,
}
USD_TO_KRW = 1370

inst = "unknown"
try:
    inst = json.loads(
        Path("/opt/ml/metadata/resource-metadata.json").read_text()
    ).get("InstanceType", "unknown")
except Exception:
    pass
rate = INSTANCE_RATES.get(inst)

setup_min = (download_time + load_time) / 60
total_min = setup_min + train_time / 60

print("=" * 60)
print("COST ANALYSIS — M8 Cosmos Reason LoRA SFT")
print("=" * 60)
print(f"Instance:          {inst}")
print(f"Resolution tier:   {TIER}")
print(f"Model restore:     {download_time/60:.1f} min")
print(f"Model load:        {load_time/60:.1f} min")
print(f"Training:          {train_time/60:.1f} min ({NUM_STEPS} steps)")
print(f"Total:             {total_min:.1f} min")
if rate is not None:
    usd = rate * total_min / 60
    print(f"Rate:              ${rate}/hr")
    print(f"Estimated cost:    ${usd:.2f} USD / {usd*USD_TO_KRW:,.0f} KRW")
    print(f"Cost per step:     ${usd/NUM_STEPS:.4f}")
else:
    print(f"Rate:              (unknown instance '{inst}' — not in rate table)")
print("=" * 60)
print("Full fine-tuning this 8.33 B model instead of LoRA would need ~123 GiB of")
print("weight+gradient+optimizer state — more than this box has in total, which is")
print("the entire reason PEFT is the method here.")


In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m08-cosmos-sft")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")
